# Step 1C — Sketch Mode

The child draws on the canvas in the app. The drawing is captured as a JPEG and sent to the backend.

```
Character selection
    └─ Canvas drawing (JPEG)
           ├─ POST /api/sketch-preview
           │      ├─ 1. Label extraction   (Flash Lite reads the sketch → 3–6 word label)
           │      ├─ 2. Safety check       (SAFE / UNSAFE)
           │      └─ 3. Illustration gen   (Gemini image model → storybook art)
           └─ POST /api/story-opening      (opening text + first image, using prop label)
```

Same pipeline as 1B — the only difference is the input comes from a canvas drawing rather than a camera.

---
## Setup

In [ ]:
import sys, os, base64, io
sys.path.insert(0, '../backend')

from dotenv import load_dotenv
load_dotenv('../backend/.env')

print('PROJECT:      ', os.environ.get('GOOGLE_CLOUD_PROJECT', '❌ not set'))
print('IMAGE_MODEL:  ', os.environ.get('IMAGE_MODEL', '❌ not set'))
print('GEMINI_API_KEY:', 'set' if os.environ.get('GEMINI_API_KEY') else '❌ not set')

---
## Initial Step — Character Selection

In [ ]:
from characters import CHARACTERS, get_character

print(f'{len(CHARACTERS)} characters available:\n')
for char in CHARACTERS.values():
    print(f'  {char.id:20s}  lang={char.language:10s}  voice={char.voice_name:15s}  {char.name}')

In [ ]:
# Pick a character
char = get_character('robot')

print(f'Name:         {char.name}')
print(f'Language:     {char.language}')
print(f'Voice:        {char.voice_name}')
print(f'Image style:  {char.image_style}')
print(f'\n--- System prompt (first 400 chars) ---')
print(char.system_prompt[:400], '...')

---
## Create a Test Sketch

Two options:
- **Option A** — draw a sketch with PIL (no file needed, good for quick testing)
- **Option B** — load an existing sketch image from disk

In [ ]:
# Option A — generate a simple sketch with PIL
from PIL import Image as PILImage, ImageDraw
from IPython.display import Image, display

# Canvas background is white (matches the app's drawing canvas)
img = PILImage.new('RGB', (400, 300), color=(255, 255, 255))
draw = ImageDraw.Draw(img)

# Draw a simple robot
draw.rectangle([150, 60, 250, 140], outline='black', width=3)    # head
draw.rectangle([130, 145, 270, 240], outline='black', width=3)   # body
draw.rectangle([100, 145, 125, 220], outline='black', width=3)   # left arm
draw.rectangle([275, 145, 300, 220], outline='black', width=3)   # right arm
draw.rectangle([150, 245, 185, 290], outline='black', width=3)   # left leg
draw.rectangle([215, 245, 250, 290], outline='black', width=3)   # right leg
draw.ellipse([165, 75, 195, 105], outline='black', width=2)      # left eye
draw.ellipse([205, 75, 235, 105], outline='black', width=2)      # right eye
draw.arc([170, 110, 230, 135], start=0, end=180, fill='black', width=2)  # smile
draw.ellipse([190, 155, 210, 175], outline='black', width=2)     # chest light

buf = io.BytesIO()
img.save(buf, format='JPEG', quality=85)
sketch_bytes = buf.getvalue()
sketch_b64 = base64.b64encode(sketch_bytes).decode()

print(f'Sketch: {img.size[0]}x{img.size[1]}px, {len(sketch_bytes):,} bytes')
display(Image(data=sketch_bytes))

In [ ]:
# Option B — load a sketch from disk
# SKETCH_PATH = '/path/to/your/sketch.jpg'
# with open(SKETCH_PATH, 'rb') as f:
#     raw = f.read()
# pil_img = PILImage.open(io.BytesIO(raw)).convert('RGB')
# buf = io.BytesIO()
# pil_img.save(buf, format='JPEG', quality=85)
# sketch_bytes = buf.getvalue()
# sketch_b64 = base64.b64encode(sketch_bytes).decode()
# print(f'Loaded: {SKETCH_PATH} ({len(sketch_bytes):,} bytes)')
# display(Image(data=sketch_bytes))

---
## Step 1 — Label Extraction

Flash Lite looks at the sketch and describes the main subject in 3–6 friendly words.

In [ ]:
from google.genai import types
from image_gen import EXTRACT_MODEL, _extract_client

print(f'Model: {EXTRACT_MODEL}\n')

label_response = await _extract_client.aio.models.generate_content(
    model=EXTRACT_MODEL,
    contents=[
        types.Part(inline_data=types.Blob(mime_type='image/jpeg', data=sketch_bytes)),
        types.Part(text=(
            'A young child drew this. In 3–6 simple, friendly words describe '
            'the main subject — e.g. \'a friendly blue dragon\', \'a big rainbow castle\', '
            '\'a red toy car\'. No punctuation at the end. Keep it imaginative and warm.'
        )),
    ],
)

label = label_response.text.strip().rstrip('.')
print(f'Label: {label!r}')

---
## Step 2 — Safety Check

In [ ]:
from image_gen import _is_safe_for_children

safe = await _is_safe_for_children(label)

print(f'Label: {label!r}')
print(f'Safe:  {safe}')

if not safe:
    print('\n❌ Would be rejected — endpoint returns HTTP 400 unsafe_content')
else:
    print('\n✅ Proceeding to illustration')

---
## Step 3 — Illustration Generation

The sketch + label go to the Gemini image model. It recreates the child's drawing as warm storybook art, keeping the same subject and composition.

In [ ]:
from image_gen import IMAGE_MODEL, _generate_from_sketch

print(f'Image model: {IMAGE_MODEL}')
print(f'Art style:   {char.image_style}')
print(f'Label hint:  {label!r}')
print()

image_b64, mime_type = await _generate_from_sketch(sketch_bytes, char.image_style, label)

print(f'Generated: {mime_type}, {len(image_b64):,} chars')
display(Image(data=base64.b64decode(image_b64)))

---
## Full `POST /api/sketch-preview`

Runs all three steps as one call — exactly what the frontend sends after the child finishes drawing.

In [ ]:
from image_gen import sketch_preview, SketchPreviewRequest

request = SketchPreviewRequest(
    sketch_data=sketch_b64,
    image_style=char.image_style,
)

result = await sketch_preview(request)

print(f'Label: {result.label!r}')
print(f'Image: {result.mime_type}, {len(result.image_data):,} chars')
display(Image(data=base64.b64decode(result.image_data)))

---
## `POST /api/story-opening` — Opening Text + First Illustration

The sketch label becomes `prop_description`. The opening story puts the child's own drawing into the narrative — it becomes a character in the story world.

In [ ]:
from image_gen import _generate_opening

prop_description = result.label

print(f'Character:  {char.name}')
print(f'Prop:       {prop_description!r}')
print()

opening_text, scene_description = await _generate_opening(
    character_name=char.name,
    character_language=char.language,
    theme='sketch',
    prop_description=prop_description,
)

print('=== STORY OPENING ===')
print(opening_text)
print('\n=== SCENE DESCRIPTION (for image gen) ===')
print(scene_description)

In [ ]:
from image_gen import generate_story_opening, StoryOpeningRequest

opening_request = StoryOpeningRequest(
    character_id=char.id,
    character_name=char.name,
    character_language=char.language,
    image_style=char.image_style,
    theme='sketch',
    prop_description=result.label,
)

opening_result = await generate_story_opening(opening_request)

print('Opening text:')
print(opening_result.opening_text)
print(f'\nScene: {opening_result.scene_description}')
print(f'Image: {opening_result.mime_type}, {len(opening_result.image_data):,} chars')
display(Image(data=base64.b64decode(opening_result.image_data)))